# ADK Travel Planner — Research Notebook

Multi-agent travel planner using Google ADK. Agents are defined and orchestrated
inline — no external servers required.

**Architecture:**
```
Request → Host Agent → Flight Agent
                    → Stay Agent
                    → Activities Agent
```

Set `DEEPSEEK_API_KEY` in your environment for real DeepSeek responses.
Without it, the notebook falls back to mock data.

In [1]:
import os
import json
from pprint import pprint

HAS_API_KEY = bool(os.getenv("DEEPSEEK_API_KEY"))

---
## Define agents inline

In [2]:
async def flight_agent(request):
    origin = request["origin"]
    dest = request["destination"]
    start = request["start_date"]
    end = request["end_date"]
    budget = request["budget"]

    if HAS_API_KEY:
        from google.adk.agents import Agent as AdkAgent
        from google.adk.models.lite_llm import LiteLlm
        from google.adk.runners import Runner
        from google.adk.sessions import InMemorySessionService
        from google.genai import types

        agent = AdkAgent(
            name="flight_agent",
            model=LiteLlm("deepseek/deepseek-chat"),
            instruction=(
                "Given a destination, travel dates, and budget, suggest 1-2 realistic flight options. "
                "Include airline name, price, and departure time. Ensure flights fit within the budget."
            )
        )
        svc = InMemorySessionService()
        runner = Runner(agent=agent, app_name="flight", session_service=svc)
        await svc.create_session(app_name="flight", user_id="u1", session_id="s1")

        prompt = (
            f"User is flying from {origin} to {dest} "
            f"from {start} to {end}, with a budget of ${budget}. "
            "Suggest 2-3 realistic flight options. For each option, include airline, departure time, return time, "
            "price, and mention if it's direct or has layovers."
        )
        msg = types.Content(role="user", parts=[types.Part(text=prompt)])
        async for event in runner.run_async(user_id="u1", session_id="s1", new_message=msg):
            if event.is_final_response():
                return {"flights": event.content.parts[0].text}
        return {"flights": "No response from agent."}

    # Mock fallback
    return {
        "flights": (
            f"1. {['Delta','United','American'][hash(dest)%3]} — ${budget-500} direct 8h\n"
            f"2. {['JetBlue','Alaska','Southwest'][hash(origin)%3]} — ${budget-300} 1 stop 10h"
        )
    }

In [3]:
async def stay_agent(request):
    dest = request["destination"]
    start = request["start_date"]
    end = request["end_date"]
    budget = request["budget"]

    if HAS_API_KEY:
        from google.adk.agents import Agent as AdkAgent
        from google.adk.models.lite_llm import LiteLlm
        from google.adk.runners import Runner
        from google.adk.sessions import InMemorySessionService
        from google.genai import types

        agent = AdkAgent(
            name="stay_agent",
            model=LiteLlm("deepseek/deepseek-chat"),
            instruction=(
                "Given a destination, travel dates, and budget, suggest 2-3 hotel or stay options. "
                "Include hotel name, price per night, and location. Ensure suggestions are within budget."
            )
        )
        svc = InMemorySessionService()
        runner = Runner(agent=agent, app_name="stay", session_service=svc)
        await svc.create_session(app_name="stay", user_id="u1", session_id="s1")

        prompt = (
            f"User is staying in {dest} from {start} to {end} "
            f"with a budget of ${budget}. Suggest stay options."
        )
        msg = types.Content(role="user", parts=[types.Part(text=prompt)])
        async for event in runner.run_async(user_id="u1", session_id="s1", new_message=msg):
            if event.is_final_response():
                return {"stays": event.content.parts[0].text}
        return {"stays": "No response."}

    return {
        "stays": (
            f"1. Grand {dest} Hotel — ${int(budget*0.3)}/night downtown\n"
            f"2. {dest} Budget Inn — ${int(budget*0.15)}/night near transit"
        )
    }

In [4]:
async def activities_agent(request):
    dest = request["destination"]
    start = request["start_date"]
    end = request["end_date"]
    budget = request["budget"]

    if HAS_API_KEY:
        from google.adk.agents import Agent as AdkAgent
        from google.adk.models.lite_llm import LiteLlm
        from google.adk.runners import Runner
        from google.adk.sessions import InMemorySessionService
        from google.genai import types

        agent = AdkAgent(
            name="activities_agent",
            model=LiteLlm("deepseek/deepseek-chat"),
            instruction=(
                "Given a destination, dates, and budget, suggest 2-3 engaging tourist or cultural activities. "
                "For each activity, provide name, a short description, price estimate, and duration in hours. "
                "Respond in plain English (not JSON). Keep it concise and well-formatted."
            )
        )
        svc = InMemorySessionService()
        runner = Runner(agent=agent, app_name="activities", session_service=svc)
        await svc.create_session(app_name="activities", user_id="u1", session_id="s1")

        prompt = (
            f"User is visiting {dest} "
            f"from {start} to {end}, with a budget of ${budget}. "
            "Suggest 2-3 activities or attractions. For each option, include activity name, estimated cost, "
            "duration, and a brief description of the experience."
        )
        msg = types.Content(role="user", parts=[types.Part(text=prompt)])
        async for event in runner.run_async(user_id="u1", session_id="s1", new_message=msg):
            if event.is_final_response():
                return {"activities": event.content.parts[0].text}
        return {"activities": "No response."}

    return {
        "activities": (
            f"1. {dest} City Tour — ${int(budget*0.1)} (3h)\n"
            f"2. Local Food Tasting — ${int(budget*0.05)} (2h)"
        )
    }

---
## Orchestrator — calls all three agents

In [5]:
async def host_agent(request):
    flights = await flight_agent(request)
    stays = await stay_agent(request)
    activities = await activities_agent(request)
    return {
        "flights": flights.get("flights", "N/A"),
        "stay": stays.get("stays", "N/A"),
        "activities": activities.get("activities", "N/A")
    }

---
## Run a trip

In [6]:
trip = {
    "origin": "New York",
    "destination": "Tokyo",
    "start_date": "2026-09-15",
    "end_date": "2026-09-25",
    "budget": 4000
}

result = await host_agent(trip)

print("=== FLIGHTS ===")
print(result["flights"])
print("\n=== STAYS ===")
print(result["stay"])
print("\n=== ACTIVITIES ===")
print(result["activities"])

Root node flight_agent was cancelled.
Failed to detach context
Traceback (most recent call last):
  File "/opt/anaconda3/envs/my_python_3_12/lib/python3.12/site-packages/opentelemetry/trace/__init__.py", line 608, in use_span
    yield span
  File "/opt/anaconda3/envs/my_python_3_12/lib/python3.12/site-packages/opentelemetry/trace/__init__.py", line 508, in start_as_current_span
    yield span
  File "/opt/anaconda3/envs/my_python_3_12/lib/python3.12/site-packages/opentelemetry/trace/__init__.py", line 443, in start_as_current_span
    yield span
  File "/opt/anaconda3/envs/my_python_3_12/lib/python3.12/site-packages/google/adk/runners.py", line 572, in _run_node_async
    yield event
GeneratorExit

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/opt/anaconda3/envs/my_python_3_12/lib/python3.12/site-packages/opentelemetry/context/__init__.py", line 143, in detach
    _RUNTIME_CONTEXT.detach(token)
  File "/opt/anaconda3/e

=== FLIGHTS ===
Based on typical flight routes from New York (JFK) to Tokyo (NRT/HND) in September 2026, here are 2 realistic options within your $4000 budget. Note that prices are estimates for typical economy/premium economy fares; actual prices vary.

### Option 1: Direct & Premium Economy (Best for comfort & time)
- **Airline:** Japan Airlines (JAL) or ANA
- **Departure:** New York JFK → Tokyo (NRT or HND) – September 15, ~11:00 AM (departure time may be evening for some direct flights)
- **Return:** Tokyo → New York – September 25, ~5:00 PM
- **Price:** ~$2,800 - $3,500 (Premium Economy)
- **Details:** Direct flight, ~13-14 hours each way. This fits well within your budget and offers a superior experience (more legroom, better meals, priority boarding) compared to standard economy.

### Option 2: Economy with One Layover (Cheaper & flexible timing)
- **Airline:** Cathay Pacific (layover in Hong Kong) or Turkish Airlines (layover in Istanbul)
- **Departure:** New York JFK → Tokyo –

Root node activities_agent was cancelled.
Failed to detach context
Traceback (most recent call last):
  File "/opt/anaconda3/envs/my_python_3_12/lib/python3.12/site-packages/opentelemetry/trace/__init__.py", line 608, in use_span
    yield span
  File "/opt/anaconda3/envs/my_python_3_12/lib/python3.12/site-packages/opentelemetry/trace/__init__.py", line 508, in start_as_current_span
    yield span
  File "/opt/anaconda3/envs/my_python_3_12/lib/python3.12/site-packages/opentelemetry/trace/__init__.py", line 443, in start_as_current_span
    yield span
  File "/opt/anaconda3/envs/my_python_3_12/lib/python3.12/site-packages/google/adk/runners.py", line 572, in _run_node_async
    yield event
GeneratorExit

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/opt/anaconda3/envs/my_python_3_12/lib/python3.12/site-packages/opentelemetry/context/__init__.py", line 143, in detach
    _RUNTIME_CONTEXT.detach(token)
  File "/opt/anacond

---
## Experiment — change the trip

In [7]:
trip2 = {
    "origin": "Dublin",
    "destination": "Bangkok",
    "start_date": "2026-11-01",
    "end_date": "2026-11-14",
    "budget": 2500
}

result2 = await host_agent(trip2)

print("=== FLIGHTS ===")
print(result2["flights"])
print("\n=== STAYS ===")
print(result2["stay"])
print("\n=== ACTIVITIES ===")
print(result2["activities"])

Root node flight_agent was cancelled.
Failed to detach context
Traceback (most recent call last):
  File "/opt/anaconda3/envs/my_python_3_12/lib/python3.12/site-packages/opentelemetry/trace/__init__.py", line 608, in use_span
    yield span
  File "/opt/anaconda3/envs/my_python_3_12/lib/python3.12/site-packages/opentelemetry/trace/__init__.py", line 508, in start_as_current_span
    yield span
  File "/opt/anaconda3/envs/my_python_3_12/lib/python3.12/site-packages/opentelemetry/trace/__init__.py", line 443, in start_as_current_span
    yield span
  File "/opt/anaconda3/envs/my_python_3_12/lib/python3.12/site-packages/google/adk/runners.py", line 572, in _run_node_async
    yield event
GeneratorExit

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/opt/anaconda3/envs/my_python_3_12/lib/python3.12/site-packages/opentelemetry/context/__init__.py", line 143, in detach
    _RUNTIME_CONTEXT.detach(token)
  File "/opt/anaconda3/e

=== FLIGHTS ===
Of course. Given the budget of $2,500 and the dates (Nov 1-14, 2026), here are 3 realistic flight options from Dublin (DUB) to Bangkok (BKK). Prices are approximate and subject to availability; booking early (at least 6-9 months ahead) often helps secure lower fares.

---

### Option 1: Low Cost / 1 Stop – Best Value
- **Airline:** Etihad Airways  
- **Outbound:** Dublin (DUB) → Abu Dhabi (AUH) → Bangkok (BKK)  
  - Departure: Nov 1, 2026 – 09:15 (DUB)  
  - Arrival: Nov 2, 2026 – 07:55 (BKK)  
- **Return:** Bangkok (BKK) → Abu Dhabi (AUH) → Dublin (DUB)  
  - Departure: Nov 14, 2026 – 23:55 (BKK)  
  - Arrival: Nov 15, 2026 – 13:15 (DUB)  
- **Price:** ~$1,100–$1,350 (Economy)  
- **Stops:** 1 layover in Abu Dhabi (each way); layover time approx. 2-4 hours.  
- **Note:** Excellent value, good service, and well within budget.

---

### Option 2: Major Airline / 1 Stop – Good Balance
- **Airline:** Emirates  
- **Outbound:** Dublin (DUB) → Dubai (DXB) → Bangkok (BKK)  
 

Root node activities_agent was cancelled.
Failed to detach context
Traceback (most recent call last):
  File "/opt/anaconda3/envs/my_python_3_12/lib/python3.12/site-packages/opentelemetry/trace/__init__.py", line 608, in use_span
    yield span
  File "/opt/anaconda3/envs/my_python_3_12/lib/python3.12/site-packages/opentelemetry/trace/__init__.py", line 508, in start_as_current_span
    yield span
  File "/opt/anaconda3/envs/my_python_3_12/lib/python3.12/site-packages/opentelemetry/trace/__init__.py", line 443, in start_as_current_span
    yield span
  File "/opt/anaconda3/envs/my_python_3_12/lib/python3.12/site-packages/google/adk/runners.py", line 572, in _run_node_async
    yield event
GeneratorExit

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/opt/anaconda3/envs/my_python_3_12/lib/python3.12/site-packages/opentelemetry/context/__init__.py", line 143, in detach
    _RUNTIME_CONTEXT.detach(token)
  File "/opt/anacond

In [10]:
print("=== FLIGHTS ===")
print(result2["flights"])

=== FLIGHTS ===
Of course. Given the budget of $2,500 and the dates (Nov 1-14, 2026), here are 3 realistic flight options from Dublin (DUB) to Bangkok (BKK). Prices are approximate and subject to availability; booking early (at least 6-9 months ahead) often helps secure lower fares.

---

### Option 1: Low Cost / 1 Stop – Best Value
- **Airline:** Etihad Airways  
- **Outbound:** Dublin (DUB) → Abu Dhabi (AUH) → Bangkok (BKK)  
  - Departure: Nov 1, 2026 – 09:15 (DUB)  
  - Arrival: Nov 2, 2026 – 07:55 (BKK)  
- **Return:** Bangkok (BKK) → Abu Dhabi (AUH) → Dublin (DUB)  
  - Departure: Nov 14, 2026 – 23:55 (BKK)  
  - Arrival: Nov 15, 2026 – 13:15 (DUB)  
- **Price:** ~$1,100–$1,350 (Economy)  
- **Stops:** 1 layover in Abu Dhabi (each way); layover time approx. 2-4 hours.  
- **Note:** Excellent value, good service, and well within budget.

---

### Option 2: Major Airline / 1 Stop – Good Balance
- **Airline:** Emirates  
- **Outbound:** Dublin (DUB) → Dubai (DXB) → Bangkok (BKK)  
 

In [9]:
print("\n=== STAYS ===")
print(result2["stay"])


=== STAYS ===
Here are three stay options in Bangkok within your $2500 budget for the November 1–14, 2026 dates (13 nights):

1. **Hotel Name:** The Peninsula Bangkok  
   **Price per night:** ~$180  
   **Location:** 333 Charoennakorn Road, Riverside (Khlong San district) – luxury riverside hotel with shuttle boat, spa, and fine dining.  
   **Total for 13 nights:** ~$2,340 (within budget)

2. **Hotel Name:** Amara Bangkok Hotel  
   **Price per night:** ~$120  
   **Location:** 180/1 Surawong Road, Silom (near Patpong, Chong Nonsi BTS) – modern boutique hotel with rooftop infinity pool and city views.  
   **Total for 13 nights:** ~$1,560 (well under budget, leaving room for activities)

3. **Hotel Name:** Ibis Styles Bangkok Khaosan Viengtai  
   **Price per night:** ~$70  
   **Location:** 56/1 Soi 2, Khaosan Road (Phra Nakhon district) – budget-friendly, colorful hotel near Khao San Road, walking distance to Grand Palace and temples.  
   **Total for 13 nights:** ~$910 (very budg

In [8]:
print("\n=== ACTIVITIES ===")
print(result2["activities"])


=== ACTIVITIES ===
Here are 3 engaging activities in Bangkok that fit within your $2500 budget for the 2-week trip.

**1. Grand Palace & Wat Phra Kaew (Temple of the Emerald Buddha)**
- **Cost:** ~$15 USD (500 THB) per person for entry fee
- **Duration:** 3 hours
- **Description:** Explore the former royal residence and the most sacred Buddhist temple in Thailand, featuring intricate gold architecture, murals, and the revered Emerald Buddha statue. A must-see cultural highlight of Bangkok.

**2. Floating Market Day Trip (Damnoen Saduak)**
- **Cost:** ~$40–$60 USD per person (includes van transport and long-tail boat ride, meals extra)
- **Duration:** 5–6 hours (half-day excursion)
- **Description:** Navigate canals filled with vendors selling fresh fruit, local snacks, and souvenirs from wooden boats. A colorful, vibrant experience that offers a glimpse into traditional Thai water-based commerce and culture.

**3. Evening Tuk-Tuk Food Tour (Chinatown & Old City)**
- **Cost:** ~$50–$70

---
## How it works

1. **Orchestrator** (`host_agent`) receives a request with origin, destination, dates, budget
2. Dispatches to three domain **sub-agents** — each uses ADK + LiteLLM + DeepSeek (or mock)
3. Each sub-agent returns structured suggestions for its domain
4. Orchestrator collects all responses and returns a combined result

The A2A (Agent-to-Agent) protocol in the real system does this over HTTP.
This notebook does it with direct function calls for simplicity and reproducibility.